# P5-02 — new datasets, fresh five-class ResNet34 U-Net

Attach the private **kaggle-p502-new-datasets** dataset and enable a GPU.
The notebook accepts either the uploaded ZIP or its unpacked contents.
Official ImageNet encoder weights are bundled; Internet is not required. This is a fresh run:
the old notebook's checkpoints have different preprocessing/training settings.

New COCO/YOLO, LunarSeg and AI4Mars additions are training-only. The inherited
validation/test scenes remain fixed and sparse; their prior results have
already been inspected. These are engineering gates, not fresh independent
qualification evidence. Gates stay mIoU >=0.50 and each class >=0.25.


In [ ]:
import csv, hashlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path
import torch, torchvision

# Edit these before saving a run.
RESUME = False
STOP_AFTER_EPOCH = 1
BATCH_SIZE = 8
IMAGE_SIZE = 384
WORKERS = 2
RESTORE_RUN_FROM = None  # set only when multiple prior run directories are attached
RUN_TEST = False
EXPORT_ZIP = False

assert torch.cuda.is_available(), "Enable a GPU in notebook settings."
print(torch.__version__, torchvision.__version__, torch.cuda.get_device_name(0))
INPUT = Path('/kaggle/input')
manifests = [p for p in INPUT.rglob('manifest.csv')
             if (p.parent / 'tools/train_downloaded_resnet.py').is_file()
             and (p.parent / 'config/gates-frozen.json').is_file()]
if manifests:
    assert len(manifests) == 1, f"Attach one training bundle: {manifests}"
    BUNDLE = manifests[0].parent
else:
    archives = list(INPUT.rglob('kaggle-p502-new-datasets.zip'))
    assert len(archives) == 1, f"Attach the training dataset ZIP: {archives}"
    BUNDLE = Path('/tmp/p502-dataset')
    marker = BUNDLE / '.extracted-from'
    archive = archives[0]
    fingerprint = f'{archive}:{archive.stat().st_size}:{archive.stat().st_mtime_ns}'
    if not marker.is_file() or marker.read_text() != fingerprint:
        if BUNDLE.exists():
            shutil.rmtree(BUNDLE)
        with zipfile.ZipFile(archive) as zipped:
            required = sum(item.file_size for item in zipped.infolist())
            assert shutil.disk_usage(BUNDLE.parent).free > required, 'Insufficient space to unpack dataset'
            print(f'Unpacking {archive.name} into {BUNDLE}', flush=True)
            zipped.extractall(BUNDLE)
        marker.write_text(fingerprint)
print('Bundle:', BUNDLE)
with (BUNDLE / 'manifest.csv').open() as stream:
    from collections import Counter
    counts = Counter(row['split'] for row in csv.DictReader(stream))
assert counts == {'train': 32104, 'validation': 746, 'test': 749}, counts
print('Records:', dict(counts))
print((BUNDLE / 'config/gates-frozen.json').read_text())
os.environ['TORCH_HOME'] = '/tmp/p502-torch-cache'
encoder = BUNDLE / 'weights/resnet34-b627a593.pth'
assert hashlib.sha256(encoder.read_bytes()).hexdigest().startswith('b627a593'), 'Encoder weight hash mismatch'
encoder_cache = Path(os.environ['TORCH_HOME']) / 'hub/checkpoints'
encoder_cache.mkdir(parents=True, exist_ok=True)
shutil.copy2(encoder, encoder_cache / encoder.name)


In [ ]:
# Install the trainer shipped with this notebook and the bundle's supporting modules.
TOOLS = Path('/tmp/p502-tools')
shutil.copytree(BUNDLE / 'tools', TOOLS, dirs_exist_ok=True)
sys.path.insert(0, str(TOOLS))
(TOOLS / 'train_downloaded_resnet.py').write_text(
    '"""Fresh RGB ResNet34 U-Net development run on the D: canonical manifest.\n'
    '\n'
    'Validation selects checkpoints using all five classes; test inference is absent.\n'
    'Frozen acceptance gates are recorded, never changed. Run with the D: venv.\n'
    '"""\n'
    'import argparse\n'
    'import copy\n'
    'import hashlib\n'
    'import json\n'
    'import os\n'
    'import random\n'
    'import math\n'
    'import time\n'
    'from collections import Counter\n'
    'from pathlib import Path\n'
    '\n'
    'import numpy as np\n'
    'from PIL import Image\n'
    'import torch\n'
    'from torch import nn\n'
    'from torch.nn import functional as F\n'
    'from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler\n'
    'import torchvision\n'
    '\n'
    'from train_perception import load_records, load_score_gates, audit_class_support, sha256_file, CLASS_NAMES\n'
    'from perception_metrics import summarize_confusion\n'
    '\n'
    '\n'
    'class Block(nn.Module):\n'
    '    def __init__(self, incoming, outgoing):\n'
    '        super().__init__()\n'
    '        self.f = nn.Sequential(nn.Conv2d(incoming, outgoing, 3, padding=1, bias=False), nn.BatchNorm2d(outgoing), nn.ReLU(True),\n'
    '                               nn.Conv2d(outgoing, outgoing, 3, padding=1, bias=False), nn.BatchNorm2d(outgoing), nn.ReLU(True))\n'
    '\n'
    '    def forward(self, x):\n'
    '        return self.f(x)\n'
    '\n'
    '\n'
    'class UNetR34(nn.Module):\n'
    '    def __init__(self, pretrained=False):\n'
    '        super().__init__()\n'
    '        r = torchvision.models.resnet34(weights=torchvision.models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None)\n'
    '        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu)\n'
    '        self.pool = r.maxpool\n'
    '        self.l1, self.l2, self.l3, self.l4 = r.layer1, r.layer2, r.layer3, r.layer4\n'
    '        self.d4, self.d3 = Block(768, 256), Block(384, 128)\n'
    '        self.d2, self.d1 = Block(192, 64), Block(128, 48)\n'
    '        self.drop = nn.Dropout2d(0.1)\n'
    '        self.head = nn.Conv2d(48, 5, 1)\n'
    '\n'
    '    def forward(self, x):\n'
    '        size = x.shape[-2:]\n'
    '        e0 = self.stem(x)\n'
    '        e1 = self.l1(self.pool(e0)); e2 = self.l2(e1); e3 = self.l3(e2); e4 = self.l4(e3)\n'
    "        up = lambda value, ref: F.interpolate(value, size=ref.shape[-2:], mode='bilinear', align_corners=False)\n"
    '        d = self.d4(torch.cat([up(e4, e3), e3], 1))\n'
    '        d = self.d3(torch.cat([up(d, e2), e2], 1))\n'
    '        d = self.d2(torch.cat([up(d, e1), e1], 1))\n'
    '        d = self.d1(torch.cat([up(d, e0), e0], 1))\n'
    "        return F.interpolate(self.head(self.drop(d)), size=size, mode='bilinear', align_corners=False)\n"
    '\n'
    '\n'
    'class Pairs(Dataset):\n'
    '    def __init__(self, rows, size, train=False):\n'
    '        self.rows, self.size, self.train = rows, size, train\n'
    '\n'
    '    def __len__(self):\n'
    '        return len(self.rows)\n'
    '\n'
    '    def image(self, row):\n'
    "        with Image.open(row['image']) as im:\n"
    "            return np.array(im.convert('RGB').resize((self.size, self.size), Image.Resampling.BILINEAR), dtype=np.float32) / 255\n"
    '\n'
    '    def __getitem__(self, i):\n'
    '        row = self.rows[i]\n'
    '        x = self.image(row)\n'
    "        with Image.open(row['mask']) as im:\n"
    "            if im.mode not in ('L', 'P'):\n"
    "                raise ValueError('Expected indexed mask')\n"
    '            y = np.array(im.resize((self.size, self.size), Image.Resampling.NEAREST), dtype=np.int64)\n'
    '        if set(np.unique(y)) - {0, 1, 2, 3, 4, 255}:\n'
    "            raise ValueError('Invalid target ID')\n"
    '        if self.train:\n'
    '            if random.random() < 0.5:\n'
    '                x, y = x[:, ::-1], y[:, ::-1]\n'
    '            x = np.clip(x * random.uniform(.85, 1.15), 0, 1)\n'
    '        return normalized_tensor(x), torch.from_numpy(y.copy())\n'
    '\n'
    '\n'
    'def normalized_tensor(x):\n'
    '    x = (x - np.array([.485, .456, .406], np.float32)) / np.array([.229, .224, .225], np.float32)\n'
    '    return torch.from_numpy(x.transpose(2, 0, 1).copy())\n'
    '\n'
    '\n'
    '@torch.inference_mode()\n'
    'def evaluate(model, loader, device):\n'
    '    model.eval()\n'
    '    matrix = np.zeros((5, 5), dtype=np.int64)\n'
    '    for x, y in loader:\n'
    "        with torch.autocast('cuda', enabled=device.type == 'cuda'):\n"
    '            logits = model(x.to(device, non_blocking=True))\n'
    '        # Evaluate at original indexed-mask resolution, preserving sparse targets.\n'
    '        for i, target in enumerate(y):\n'
    "            pred = F.interpolate(logits[i:i+1].float(), size=target.shape, mode='bilinear', align_corners=False).argmax(1)[0].cpu().numpy()\n"
    '            target = target.numpy()\n'
    '            valid = target != 255\n'
    '            matrix += np.bincount(target[valid] * 5 + pred[valid], minlength=25).reshape(5, 5)\n'
    '    return summarize_confusion(matrix, CLASS_NAMES), matrix\n'
    '\n'
    '\n'
    'class Validation(Pairs):\n'
    '    def __getitem__(self, i):\n'
    "        with Image.open(self.rows[i]['mask']) as im:\n"
    "            if im.mode not in ('L', 'P'):\n"
    "                raise ValueError('Expected indexed mask')\n"
    '            target = torch.from_numpy(np.array(im, dtype=np.int64))\n'
    '        return normalized_tensor(self.image(self.rows[i])), target\n'
    '\n'
    '\n'
    'def collate(batch):\n'
    '    return torch.stack([x for x, _ in batch]), [y for _, y in batch]\n'
    '\n'
    '\n'
    'def masked_dice(logits, target):\n'
    '    valid = target != 255\n'
    '    probs = logits.float().softmax(1) * valid[:, None]\n'
    '    truth = F.one_hot(target.clamp(max=4), 5).permute(0, 3, 1, 2).float() * valid[:, None]\n'
    '    intersect = (probs * truth).sum((0, 2, 3))\n'
    '    denominator = probs.sum((0, 2, 3)) + truth.sum((0, 2, 3))\n'
    '    present = truth.sum((0, 2, 3)) > 0\n'
    '    return (1 - (2 * intersect + 1) / (denominator + 1))[present].mean()\n'
    '\n'
    '\n'
    'def main():\n'
    '    p = argparse.ArgumentParser(description=__doc__)\n'
    "    p.add_argument('--corpus', type=Path, required=True)\n"
    "    p.add_argument('--gates', type=Path, required=True)\n"
    "    p.add_argument('--output', type=Path, required=True)\n"
    "    p.add_argument('--epochs', type=int, default=40)\n"
    "    p.add_argument('--batch-size', type=int, default=2)\n"
    "    p.add_argument('--size', type=int, default=320)\n"
    "    p.add_argument('--workers', type=int, default=0, help='parallel training data-loader processes')\n"
    "    p.add_argument('--resume', action='store_true')\n"
    "    p.add_argument('--stop-after-epoch', type=int, help='finish a checkpointed chunk while retaining the full epoch/LR budget')\n"
    '    args = p.parse_args()\n'
    '    if args.workers < 0:\n'
    "        p.error('--workers must be nonnegative')\n"
    '    if not torch.cuda.is_available():\n'
    "        raise RuntimeError('CUDA unavailable; refusing accidental slow CPU run')\n"
    '    torch.set_num_threads(4)\n'
    '    seed = 20261110\n'
    '    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)\n'
    "    records = load_records(args.corpus / 'manifest.csv', args.corpus, args.corpus / 'splits.json', args.corpus / 'sources.json')\n"
    '    support = audit_class_support(records)\n'
    '    gates = load_score_gates(args.gates, CLASS_NAMES)\n'
    "    print('PREFLIGHT PASS', {s: len(r) for s, r in records.items()}, 'support', support, flush=True)\n"
    '    args.output.mkdir(parents=True, exist_ok=args.resume)\n'
    "    identity = dict(configuration={k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items() if k not in ('resume', 'stop_after_epoch')},\n"
    "                    hashes={name: sha256_file(args.corpus / name) for name in ['manifest.csv', 'sources.json', 'splits.json']},\n"
    '                    gates_sha256=sha256_file(args.gates), trainer_sha256=sha256_file(Path(__file__)))\n'
    "    data_hashes = {path: sha256_file(Path(path)) for path in sorted({r[k] for rs in records.values() for r in rs for k in ['image', 'mask']})}\n"
    "    identity['data_sha256'] = hashlib.sha256(json.dumps(data_hashes, sort_keys=True).encode()).hexdigest()\n"
    "    config = dict(identity=identity, model='UNetR34', preprocessing='RGB /255, ImageNet mean/std; bilinear input; nearest masks',\n"
    '                  class_names=CLASS_NAMES, ignore_id=255, seed=seed, gates=gates, class_support=support,\n'
    "                  loss='inverse-sqrt weighted cross-entropy + masked Dice',\n"
    "                  learning_rate=dict(encoder=.00025, decoder=.0005, schedule='one-epoch warmup, cosine decay'),\n"
    '                  torch_version=torch.__version__, torchvision_version=torchvision.__version__, test_predictions_inspected=False)\n'
    "    (args.output / 'run-config.json').write_text(json.dumps(config, indent=2) + '\\n')\n"
    "    (args.output / 'data-hashes.json').write_text(json.dumps(data_hashes, indent=2) + '\\n')\n"
    "    device = torch.device('cuda')\n"
    '    model = UNetR34(pretrained=not args.resume).to(device)\n'
    '    encoder = [p for module in (model.stem, model.l1, model.l2, model.l3, model.l4) for p in module.parameters()]\n'
    '    encoder_ids = {id(p) for p in encoder}\n'
    '    decoder = [p for p in model.parameters() if id(p) not in encoder_ids]\n'
    '    opt = torch.optim.AdamW([dict(params=encoder, lr=.00025), dict(params=decoder, lr=.0005)], weight_decay=.0001)\n'
    "    scaler = torch.amp.GradScaler('cuda')\n"
    '    ema = copy.deepcopy(model).eval()\n'
    '    for param in ema.parameters():\n'
    '        param.requires_grad_(False)\n'
    "    weights = np.clip(np.sqrt(sum(support['train']) / (5 * np.array(support['train']))), .25, 20)\n"
    '    criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=device), ignore_index=255)\n'
    "    sources = Counter(r['source_id'] for r in records['train'])\n"
    "    sampling = [sources[r['source_id']] ** -.5 for r in records['train']]\n"
    '    generator = torch.Generator().manual_seed(seed)\n'
    '    sampler = WeightedRandomSampler(sampling, len(sampling), replacement=True, generator=generator)\n'
    '    worker_rng = torch.Generator()\n'
    "    train = DataLoader(Pairs(records['train'], args.size, True), batch_size=args.batch_size,\n"
    '                       sampler=sampler, num_workers=args.workers, pin_memory=True,\n'
    '                       generator=worker_rng)\n'
    "    validation = DataLoader(Validation(records['validation'], args.size), batch_size=args.batch_size,\n"
    '                            collate_fn=collate, pin_memory=True)\n'
    '    start, best, history = 0, -1., []\n'
    '    if args.resume:\n'
    "        saved = torch.load(args.output / 'last.pt', map_location='cpu', weights_only=False)\n"
    "        if saved['identity'] != identity:\n"
    "            raise ValueError('Resume identity differs')\n"
    "        if not (args.output / 'best.pt').is_file():\n"
    "            raise ValueError('Missing original best.pt')\n"
    "        model.load_state_dict(saved['model']); ema.load_state_dict(saved['ema']); opt.load_state_dict(saved['optimizer']); scaler.load_state_dict(saved['scaler'])\n"
    "        start, best, history = saved['epoch'], saved['best'], saved['history']\n"
    "        random.setstate(saved['python_rng']); np.random.set_state(saved['numpy_rng']); torch.set_rng_state(saved['torch_rng'])\n"
    "        torch.cuda.set_rng_state_all(saved['cuda_rng']); generator.set_state(saved['sampler_rng'])\n"
    '    last_epoch = min(args.epochs, args.stop_after_epoch or args.epochs)\n'
    '    if last_epoch <= start:\n'
    "        raise ValueError('Chunk must end after the resumed epoch')\n"
    '    for epoch in range(start, last_epoch):\n'
    '        started = time.monotonic()\n'
    '        worker_rng.manual_seed(seed + epoch)\n'
    '        model.train(); loss_sum = 0.; used = 0\n'
    '        for step, (x, y) in enumerate(train):\n'
    '            global_step = epoch * len(train) + step\n'
    '            warmup = len(train)\n'
    '            total = args.epochs * len(train)\n'
    '            factor = ((global_step + 1) / warmup if global_step < warmup else\n'
    '                      .5 * (1 + math.cos(math.pi * (global_step - warmup) / max(total - warmup, 1))))\n'
    '            for group, base_lr in zip(opt.param_groups, [.00025, .0005]):\n'
    "                group['lr'] = base_lr * max(factor, .01)\n"
    '            if not (y != 255).any():\n'
    '                continue\n'
    '            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)\n'
    '            opt.zero_grad(set_to_none=True)\n'
    "            with torch.autocast('cuda', dtype=torch.float16):\n"
    '                logits = model(x)\n'
    '            loss = criterion(logits.float(), y) + masked_dice(logits, y)\n'
    '            if not torch.isfinite(loss):\n'
    "                raise RuntimeError('Nonfinite loss')\n"
    '            scaler.scale(loss).backward(); scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 2.)\n'
    '            scaler.step(opt); scaler.update()\n'
    '            with torch.no_grad():\n'
    '                current = model.state_dict()\n'
    '                decay = min(.999, (1 + global_step) / (10 + global_step))\n'
    '                for key, value in ema.state_dict().items():\n'
    '                    source = current[key]\n'
    '                    if value.is_floating_point():\n'
    '                        value.mul_(decay).add_(source, alpha=1-decay)\n'
    '                    else:\n'
    '                        value.copy_(source)\n'
    '            loss_sum += loss.item(); used += 1\n'
    '            if step % 100 == 0:\n'
    "                print(f'epoch {epoch+1} batch {step}/{len(train)} loss {loss.item():.4f}', flush=True)\n"
    '        metrics, matrix = evaluate(ema, validation, device)\n'
    "        passed = (metrics['miou'] is not None and metrics['miou'] >= gates['minimum_miou']\n"
    "                  and all(metrics['per_class'][name]['support_pixels'] > 0\n"
    "                          and metrics['per_class'][name]['iou'] is not None\n"
    "                          and metrics['per_class'][name]['iou'] >= gates['minimum_per_class_iou'][name]\n"
    '                          for name in CLASS_NAMES))\n'
    '        row = dict(epoch=epoch+1, seconds=round(time.monotonic() - started, 1),\n'
    '                   train_loss=loss_sum/max(used, 1), metrics=metrics, confusion=matrix.tolist(),\n'
    '                   validation_gate_pass=passed, test_predictions_inspected=False)\n'
    '        history.append(row)\n'
    "        if metrics['miou'] > best:\n"
    "            best = metrics['miou']\n"
    "            torch.save(dict(state_dict=ema.state_dict(), epoch=epoch+1, model='UNetR34', class_names=CLASS_NAMES,\n"
    "                            image_size=args.size, preprocessing=config['preprocessing']), args.output / 'best.pt')\n"
    "            (args.output / 'validation-metrics.json').write_text(json.dumps(\n"
    "                dict(result='VALIDATION_ONLY', **row), indent=2) + '\\n')\n"
    '        state = dict(model=model.state_dict(), ema=ema.state_dict(), optimizer=opt.state_dict(), scaler=scaler.state_dict(),\n'
    '                     epoch=epoch+1, best=best, history=history, identity=identity, python_rng=random.getstate(), numpy_rng=np.random.get_state(),\n'
    '                     torch_rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state_all(), sampler_rng=generator.get_state())\n'
    "        torch.save(state, args.output / 'last.tmp.pt'); (args.output / 'last.tmp.pt').replace(args.output / 'last.pt')\n"
    "        (args.output / 'history.json').write_text(json.dumps(history, indent=2) + '\\n')\n"
    '        print(json.dumps(row), flush=True)\n'
    "    print(f'Completed epoch {last_epoch}/{args.epochs}. Test predictions remain uninspected.', flush=True)\n"
    '\n'
    '\n'
    "if __name__ == '__main__':\n"
    '    main()\n'
    , encoding='utf-8')


## Train — no test predictions

The input is RGB, normalized using ImageNet mean/std. Images use bilinear
resizing; indexed masks use nearest-neighbor. Sparse validation masks retain
their stored resolution for scoring. No TTA, calibration, or copy-paste is used.
Checkpoint selection uses all five classes. A successful training process is
not a passing validation result: inspect `validation_gate_pass` below.

For an interrupted run, restore its whole output directory at the same RUN
path, set RESUME=True and preserve input paths, code and configuration.
The first chunk ends at epoch 1 of a fixed 40-epoch schedule. Use its measured
epoch time to choose a stop epoch that fits the next session. Keep the original
training dataset version and upload layout, attach the previous notebook output,
set RESUME=True and raise STOP_AFTER_EPOCH. Repeat until epoch 40.


In [ ]:
RUN = Path('/kaggle/working/p502-new-datasets-run-01')
assert 1 <= STOP_AFTER_EPOCH <= 40
if RESUME and not (RUN / 'last.pt').is_file():
    candidates = sorted({p.parent for p in INPUT.rglob('last.pt')
                         if (p.parent / 'best.pt').is_file()
                         and (p.parent / 'run-config.json').is_file()})
    preferred = [p for p in candidates if p.name == RUN.name]
    restore = Path(RESTORE_RUN_FROM) if RESTORE_RUN_FROM else (preferred[0] if len(preferred) == 1 else (candidates[0] if len(candidates) == 1 else None))
    assert restore is not None, f'Set RESTORE_RUN_FROM to one previous run directory: {candidates}'
    assert not RUN.exists(), f'Remove or move the incomplete run at {RUN} before restoring'
    shutil.copytree(restore, RUN)
CORPUS = Path('/tmp/p502-corpus')
command = [sys.executable, '-u', str(TOOLS / 'train_downloaded_resnet.py'),
           '--corpus', str(CORPUS),
           '--gates', str(BUNDLE / 'config/gates-frozen.json'),
           '--output', str(RUN), '--epochs', '40', '--batch-size', str(BATCH_SIZE),
           '--size', str(IMAGE_SIZE), '--workers', str(WORKERS),
           '--stop-after-epoch', str(STOP_AFTER_EPOCH)]
# The portable package puts policy JSONs in config/; create a lightweight
# canonical corpus directory with stable absolute paths, without copying images.
CORPUS.mkdir(exist_ok=True)
portable = list(csv.DictReader((BUNDLE / 'manifest.csv').open()))
for row in portable:
    for field in ('image', 'mask'):
        row[field] = str(BUNDLE / row[field])
with (CORPUS / 'manifest.csv').open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=['image','mask','source_id','terrain_id','split'])
    writer.writeheader(); writer.writerows(portable)
for name in ('sources.json', 'splits.json'):
    shutil.copy2(BUNDLE / 'config' / name, CORPUS / name)
if RESUME:
    command.append('--resume')
print(' '.join(command))
subprocess.run(command, check=True)


In [ ]:
validation = json.loads((RUN / 'validation-metrics.json').read_text())
history = json.loads((RUN / 'history.json').read_text())
from statistics import median
seconds_per_epoch = median(row['seconds'] for row in history[-3:])
print('Completed epoch:', history[-1]['epoch'], '/ 40')
print(f'Recent epoch time: {seconds_per_epoch / 60:.1f} minutes')
print(f'Estimated remaining training time: {(40 - history[-1]["epoch"]) * seconds_per_epoch / 3600:.1f} hours')
print('Selected epoch:', validation['epoch'])
print('Validation gate passed:', validation['validation_gate_pass'])
print(json.dumps(validation['metrics'], indent=2))


## Visual check — same inference as the validation metric

Gray=regolith, red=bedrock, blue=rock, orange=crater, black=shadow.
Blank truth pixels are ignored, not background ground truth. Predictions over
ignored pixels are shown but do not contribute to IoU.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from PIL import Image
from train_downloaded_resnet import UNetR34, Validation
from train_perception import load_records
from torch.nn import functional as F

records = load_records(CORPUS / 'manifest.csv', CORPUS,
                       CORPUS / 'splits.json', CORPUS / 'sources.json')
ck = torch.load(RUN / 'best.pt', map_location='cpu', weights_only=True)
net = UNetR34().cuda().eval()
net.load_state_dict(ck['state_dict'])
view = Validation(records['validation'], ck['image_size'])
rare = [i for i, r in enumerate(records['validation'])
        if np.isin(np.asarray(Image.open(r['mask'])), [1,3]).any()]
indices = list(dict.fromkeys(rare + [0, len(view)//2, len(view)-1]))[:6]
cmap = ListedColormap(['#999999','#e41a1c','#377eb8','#ff7f00','#000000'])
fig, axes = plt.subplots(len(indices), 3, figsize=(12, 3*len(indices)), squeeze=False)
for ax, i in zip(axes, indices):
    x, target = view[i]
    with torch.inference_mode(), torch.autocast('cuda'):
        logits = net(x[None].cuda())
    pred = F.interpolate(logits.float(), size=target.shape, mode='bilinear', align_corners=False).argmax(1)[0].cpu().numpy()
    truth = target.numpy().astype(float); truth[truth == 255] = np.nan
    ax[0].imshow(Image.open(records['validation'][i]['image'])); ax[0].set_title(f'Validation {i}')
    ax[1].imshow(truth, cmap=cmap, vmin=0, vmax=4); ax[1].set_title('Truth — blank=ignore')
    ax[2].imshow(pred, cmap=cmap, vmin=0, vmax=4); ax[2].set_title('Prediction')
    for item in ax: item.axis('off')
plt.tight_layout(); plt.show()
del net
torch.cuda.empty_cache()


## Final test — leave disabled during development

Choose the final model using validation, freeze the pipeline, then run once.
The evaluator requires passing validation, unchanged input/code/gate hashes,
and refuses to overwrite an existing test result. Previously inspected test
scenes remain historical engineering evidence; they are not a new blind test.


In [ ]:
if RUN_TEST and (history[-1]['epoch'] != 40 or not validation['validation_gate_pass']):
    print('Test deferred until epoch 40 and a passing validation result.')
elif RUN_TEST and (RUN / 'test-metrics.json').is_file():
    print('Existing test result:', RUN / 'test-metrics.json')
elif RUN_TEST:
    test = subprocess.run([sys.executable, '-u', str(TOOLS / 'evaluate_downloaded_resnet.py'),
                           '--corpus', str(CORPUS), '--gates', str(BUNDLE / 'config/gates-frozen.json'),
                           '--run', str(RUN)])
    if test.returncode and not (RUN / 'test-metrics.json').is_file():
        test.check_returncode()


In [ ]:
with (RUN / 'environment.txt').open('w') as stream:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=stream, check=True)
if EXPORT_ZIP:
    output = Path('/kaggle/working/p502-new-datasets-delivery.zip')
    with zipfile.ZipFile(output, 'w', allowZip64=True) as delivery:
        for folder, prefix in [(RUN, 'run'), (TOOLS, 'tools'), (BUNDLE / 'config', 'config')]:
            for path in sorted(folder.iterdir()):
                if path.is_file():
                    delivery.write(path, f'{prefix}/{path.name}')
        delivery.write(BUNDLE / 'manifest.csv', 'manifest.csv')
    print(output)
